# CedarBridge Learning Studio
## C4 Custom ReAct Agent with Configurable Personas
EECE 503P/798S · Agentic Systems · Fall 2026

This project extends [C3](https://github.com/Jiany-S/C3) with a manually written LangGraph loop, two role-specific personas, and controlled live API experiments. The business PDF and summary are reused unchanged. The C3 submission remains separate.

Run from the complete C4 project folder with Python 3.11+ and `requirements-dev.txt` installed. The notebook imports and displays the same code used by the app. Its default execution reads saved real API evidence and runs offline tests; it does not make paid requests or launch a server. Live requests and UI launch are opt-in.

## 1 Use case
**Goal:** help students choose coaching, estimate published-price costs, and request staff follow-up or record feedback. **Audience:** university students working on their own software projects.

**Tools:** `lookup_service`, `calculate_quote`, `record_customer_interest`, and `record_feedback`. The first two add read/calculation actions to C3's recording tools. Quotes are planning estimates, not availability or booking confirmations.

**Constraints:** only published business facts; no calendar, email sender, payments, university authority, grade guarantees, or ghostwriting. Records are local JSONL. Conversation memory covers recent messages within one session.

In [1]:
from pathlib import Path
import inspect, json, os, subprocess, sys
from dataclasses import asdict
from IPython.display import display, Markdown, Code
from dotenv import load_dotenv
import pandas as pd
load_dotenv('.env', override=False)
from personas import PERSONAS, PromptOptions
from prompts import INSTRUCTIONS, EXAMPLES, build_messages
from tools import load_business, Tools, TOOL_SCHEMAS
from model import OpenAIModel
from react_agent import ReActAgent, parse_final
knowledge=load_business()
display(Markdown(Path('me/business_summary.txt').read_text(encoding='utf-8')))

CedarBridge Learning Studio is a fictional, independent student project coaching business in Beirut, Lebanon. We help university students learn to plan, build, test, and explain their own software projects. We are not affiliated with AUB or any university and cannot give official academic advice or approve course choices.

Project Clarity: one 30-minute online scoping consultation, USD 15 per student. Prototype Clinic: one 60-minute online debugging and testing coaching session, USD 30 per student. Team Sprint: one 90-minute online project planning workshop for up to four students, USD 60 per team. Prices are fictional teaching data, not real offers. No other prices, discounts, or bundles are published.

Sessions take place online in English or Arabic. Service hours are Monday to Friday, 10:00-18:00, Asia/Beirut. Customers can request a consultation by sharing their name, email, and a short description of their project and desired service. Staff aim to respond within two business days. A recorded lead is a request for follow-up, not a confirmed appointment, payment, or sent email. The assistant has no live calendar or payment access.

Team: Maya Nassar (learning), Karim Haddad (engineering), Rania Saleh (coordination). All are fictional. Students bring their own work. Each session focuses on one concrete blocker and ends with a short next-step checklist. We explain tradeoffs and review tests with the student; we do not ghostwrite graded submissions, guarantee grades, or impersonate university staff. We provide coaching rather than accredited qualifications.

There is no published refund policy, weekend service, scholarship, internship placement, certification, or guaranteed admission offer. Availability on a specific date is unknown until staff confirm it. The assistant must acknowledge missing information and record the unanswered question using record_feedback. Suggestions and complaints are recorded with the same tool. Contact requests use record_customer_interest only after the customer supplies name, email, and notes and asks for follow-up.

Contact details and notes are saved locally as JSONL lead records. Feedback and execution traces are also stored locally. No email is sent and no external CRM is connected. Use invented names and example.com addresses for demonstrations. Do not provide sensitive information. The prototype has no automatic retention or deletion service; its operator must remove local logs when no longer needed.


## 2 Persona design and matching configuration
Mira guides service selection; Sami reviews a technical blocker and preparation needs. Both use the same factual sources, permissions, and tools. Tone and task emphasis change through the persona prompt. Temperature and penalties tune generation but do not create expertise or guarantee a particular tone.

In [2]:
for key,p in PERSONAS.items():
    display(Markdown(f"### {p.name} — {p.role}\n" + p.traits.replace('\n','  \n')))
    print(json.dumps(p.generation.options(),indent=2))

### Mira — Service guide
Task: help a student choose an appropriate coaching service and take a manageable next step.  
Tone: welcoming, patient, reassuring without promising outcomes.  
Vocabulary: plain language; explain project terms briefly.  
Response style: short conversational paragraphs; offer one concrete next step.  
Values: accessibility, student ownership, practical encouragement.

{
  "model": "gpt-4.1-mini",
  "temperature": 0.6,
  "top_p": 1.0,
  "frequency_penalty": 0.0,
  "presence_penalty": 0.2,
  "max_completion_tokens": 650
}


### Sami — Project reviewer
Task: assess a student's blocker, scope, and testing needs before recommending coaching.  
Tone: calm, precise, candid, and respectful.  
Vocabulary: requirements, reproducible bug, test case, constraint; explain unfamiliar terms.  
Response style: concise assessment followed by a short checklist when useful.  
Values: correctness, explicit assumptions, realistic scope, student ownership.

{
  "model": "gpt-4.1-mini",
  "temperature": 0.1,
  "top_p": 1.0,
  "frequency_penalty": 0.0,
  "presence_penalty": 0.0,
  "max_completion_tokens": 650
}


## 3 Dedicated instructions and worked examples
Persona describes who the assistant is. These instructions define the input, output, workflow, quantities, missing fields, and action boundaries. Each numbered instruction has one operational purpose. The examples are illustrative formats, not claimed executions.

In [3]:
print(INSTRUCTIONS)
print(EXAMPLES)
display(Code(inspect.getsource(build_messages),language='python'))

INSTRUCTIONS
Input: the latest customer message, recent user/assistant history, business sources, and tool observations for this turn.
Output: a native tool call when an action is needed, or the final JSON object defined in the protocol.
1. Identify the student's current request and reuse relevant details already supplied in this conversation.
2. For a service recommendation or a service price/duration question, call lookup_service first. Use all to compare services.
3. For each requested total across people or sessions, call calculate_quote with the customer's exact service, people, and sessions. Team capacity changes billing units, not the requested number of people: five students remain five, not four. For a two-session total, use sessions=2 rather than doubling a one-session quote mentally. Clarify missing quantities rather than guessing. In comparisons, label each returned quote and preserve its quantities in the final answer.
4. For a requested staff follow-up, collect the studen

def build_messages(knowledge, persona, options, history, message):
    options = options or PromptOptions()
    p = PERSONAS[persona]
    behavior = (INSTRUCTIONS if options.detailed else
                "INSTRUCTIONS\nHelp the customer using the business profile and tools. Return the required final JSON.")
    behavior += "\n" + EXAMPLES if options.few_shot else ""
    behavior += ("\nThink step by step privately before selecting an action or answering; return only the brief public decision summary and answer."
                 if options.cot else "\nSelect the next action or answer directly, without a step-by-step reasoning prompt.")
    system = (PROTOCOL + json.dumps(TOOL_SCHEMAS) + f"\nPERSONA\nYou are {p.name}, the {p.role}.\n{p.traits}\n"
              + "BUSINESS SOURCES\n" + knowledge)
    if options.instruction_role == "system":
        system += "\n" + behavior
        current = message
    else:
        current = behavior + "\nEND OF INSTRUCTIONS\nCUSTOMER MESSAGE\n" + message
    return [{"role": "system", "content": system}, *history, {"role": "user", "content": current}]

## 4 Manual ReAct graph
Only LangGraph's `StateGraph`, nodes, edges, and conditional routing are used. There is no prebuilt executor, `create_react_agent`, `ToolNode`, or built-in agent runner.

`model → act → observe → model` repeats when native tool calls are present. A response without calls goes to `finalize`, which parses the required JSON. Invalid final formatting goes through `repair`; the step budget bounds the entire turn. The model decides which action to request; Python validates and executes it. Brief public decision summaries explain results without exposing or claiming access to private chain-of-thought.

In [4]:
class NoNetworkModel:
    mode='mock'
    def complete(self,*args):
        raise RuntimeError('Graph inspection does not call a model.')
graph_agent=ReActAgent(NoNetworkModel(),log_dir=Path('tmp/notebook_graph'))
print(graph_agent.graph.get_graph().draw_mermaid())
for method in [ReActAgent.build_graph,ReActAgent.model_node,ReActAgent.next_node,
               ReActAgent.action_node,ReActAgent.observation_node,parse_final]:
    display(Code(inspect.getsource(method),language='python'))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	act(act)
	observe(observe)
	finalize(finalize)
	repair(repair)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	act --> observe;
	finalize -. &nbsp;end&nbsp; .-> __end__;
	finalize -.-> repair;
	model -. &nbsp;end&nbsp; .-> __end__;
	model -.-> act;
	model -.-> finalize;
	observe --> model;
	repair --> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



def build_graph(self):
        graph = StateGraph(AgentState)
        graph.add_node("model", self.model_node)
        graph.add_node("act", self.action_node)
        graph.add_node("observe", self.observation_node)
        graph.add_node("finalize", self.final_node)
        graph.add_node("repair", self.repair_node)
        graph.add_edge(START, "model")
        graph.add_conditional_edges("model", self.next_node,
                                    {"act": "act", "finalize": "finalize", "end": END})
        graph.add_edge("act", "observe")
        graph.add_edge("observe", "model")
        graph.add_conditional_edges("finalize", lambda s: "end" if s["answer"] or s["error"] else "repair",
                                    {"end": END, "repair": "repair"})
        graph.add_edge("repair", "model")
        return graph.compile()

def model_node(self, state):
        if state["steps"] >= self.max_steps:
            return {"error": "The assistant reached its step limit before completing the response."}
        try:
            result = self.model.complete(state["messages"], self.generation)
            msg = result["message"]
            calls = msg.get("tool_calls", [])
            if not isinstance(calls, list) or len(calls) > 4:
                raise ModelError("The model returned an invalid number of tool calls.")
            ids = set()
            for call in calls:
                if (not isinstance(call, dict) or not isinstance(call.get("id"), str)
                        or call["id"] in ids or call.get("type") != "function"
                        or not call["id"] or not isinstance(call.get("function"), dict)
                        or not isinstance(call["function"].get("name"), str)
                        or not isinstance(call["function"].get("arguments"), str)):
                    raise ModelError("The model returned malformed tool calls.")
                ids.add(call["id"])
            return {"messages": [*state["messages"], msg], "pending": calls,
                    "error": ("Step limit reached; the last requested action was not executed."
                              if calls and state["steps"] + 1 >= self.max_steps else ""),
                    "steps": state["steps"] + 1, "requests": [*state["requests"], result]}
        except ModelError as exc:
            return {"error": str(exc)}

def next_node(self, state):
        if state["error"]:
            return "end"
        # Keep one model call available to read observations after any action.
        return "act" if state["pending"] else "finalize"

def action_node(self, state):
        observations, events, cache = [], list(state["events"]), dict(state["cache"])
        for call in state["pending"]:
            name = call["function"].get("name", "")
            arguments = call["function"].get("arguments", "")
            try:
                canonical = json.dumps(json.loads(arguments), sort_keys=True, separators=(",", ":"))
            except (ValueError, TypeError):
                canonical = str(arguments)
            key = name + ":" + canonical
            reused = key in cache
            observation = cache[key] if reused else self.tools.execute(name, arguments, state["user_text"])
            cache[key] = observation
            observations.append({"role": "tool", "tool_call_id": call["id"],
                                 "content": json.dumps(observation, ensure_ascii=False)})
            events.append({"tool_call_id": call["id"], "tool": name, "arguments": arguments,
                           "observation": observation, "reused": reused})
        return {"observations": observations, "events": events, "cache": cache}

def observation_node(self, state):
        """Return every execution result with the original tool-call ID, then loop."""
        return {"messages": [*state["messages"], *state["observations"]], "pending": [], "observations": []}

def parse_final(message):
    """Accept a final answer only after explicit protocol and type checks."""
    if message.get("tool_calls"):
        raise ValueError("Tool calls must be observed before a final answer.")
    obj = json.loads(message.get("content") or "")
    if not isinstance(obj, dict) or set(obj) != {"answer", "decision_summary"}:
        raise ValueError("Final JSON requires answer and decision_summary.")
    if not all(isinstance(obj[k], str) and obj[k].strip() for k in obj):
        raise ValueError("Final fields must be nonempty strings.")
    if len(obj["decision_summary"]) > 800:
        raise ValueError("The public decision summary must be brief.")
    return obj

## 5 Tools and returned observations
The function schemas are supplied both in the system prompt and the model's tool declarations. Tool results use the original `tool_call_id`. Contact provenance, argument types, sizes, and storage errors are checked in Python. Success is taken from the returned observation, not inferred from a requested call.

In [5]:
print(json.dumps(TOOL_SCHEMAS,indent=2))
display(Code(inspect.getsource(Tools.execute),language='python'))
display(Code(inspect.getsource(Tools.calculate_quote),language='python'))

[
  {
    "type": "function",
    "function": {
      "name": "lookup_service",
      "description": "Read published service facts before recommending a service or answering a service price/duration question.",
      "strict": true,
      "parameters": {
        "type": "object",
        "properties": {
          "service": {
            "type": "string",
            "description": "Exact service name or all for a comparison"
          }
        },
        "required": [
          "service"
        ],
        "additionalProperties": false
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "calculate_quote",
      "description": "Calculate a published-price estimate for people and sessions. This is not a booking.",
      "strict": true,
      "parameters": {
        "type": "object",
        "properties": {
          "service": {
            "type": "string",
            "enum": [
              "Project Clarity",
              "Prototype Clinic",
             

def execute(self, name, arguments, user_text):
        schema = next((d["function"] for d in TOOL_SCHEMAS if d["function"]["name"] == name), None)
        if schema is None:
            return {"ok": False, "error": "Unknown tool."}
        try:
            args = json.loads(arguments)
        except (ValueError, TypeError):
            return {"ok": False, "error": "Arguments must be valid JSON."}
        if not isinstance(args, dict) or set(args) != set(schema["parameters"]["required"]):
            return {"ok": False, "error": "Missing or unexpected tool fields."}
        for key, value in args.items():
            expected = schema["parameters"]["properties"][key]["type"]
            if (expected == "string" and not isinstance(value, str)) or (expected == "integer" and type(value) is not int):
                return {"ok": False, "error": "Wrong argument type."}
        if name == "record_customer_interest":
            for field in ("name", "email"):
                if not args[field].strip() or args[field].strip().casefold() not in user_text.casefold():
                    return {"ok": False, "error": f"Ask the customer for their {field}; it was not supplied."}
        return getattr(self, name)(**args)

def calculate_quote(self, service, people, sessions):
        if service not in CATALOG or any(type(v) is not int for v in (people, sessions)):
            return {"ok": False, "error": "Use an exact service name and integer people/sessions."}
        if not 1 <= people <= 100 or not 1 <= sessions <= 20:
            return {"ok": False, "error": "People must be 1–100; sessions must be 1–20."}
        item = CATALOG[service]
        units = math.ceil(people / item["capacity"])
        return {"ok": True, "service": service, "people": people, "sessions": sessions,
                "billing_units_per_session": units, "total_usd": units * item["price_usd"] * sessions,
                "basis": "Published prices; no discounts assumed", "booking_confirmed": False}

## 6 Actual multi-action example
The following is a saved real API trial, not an offline simulation. It shows the actual requested arguments, successful observations, and final reply. `trials/` contains the other conditions and repetitions, including failures.

In [6]:
example=json.loads(Path('evidence/experiments/trials/mira_default__team_quote__1.json').read_text(encoding='utf-8'))
for trace in example['traces']:
    for event in trace['events']:
        print(event['tool'],event['arguments'])
        print(json.dumps(event['observation'],indent=2))
    print('Final answer:',trace['answer'])
    print('Model calls:',trace['steps'],'| Error:',trace['error'] or 'none')

lookup_service {"service":"all"}
{
  "ok": true,
  "services": {
    "Project Clarity": {
      "minutes": 30,
      "price_usd": 15,
      "billing": "per student",
      "capacity": 1,
      "purpose": "Scope an idea and choose manageable next steps."
    },
    "Prototype Clinic": {
      "minutes": 60,
      "price_usd": 30,
      "billing": "per student",
      "capacity": 1,
      "purpose": "Work through debugging and testing blockers."
    },
    "Team Sprint": {
      "minutes": 90,
      "price_usd": 60,
      "billing": "per team",
      "capacity": 4,
      "purpose": "Plan a project with a team of up to four students."
    }
  },
  "source": "CedarBridge published profile"
}
calculate_quote {"service":"Team Sprint","people":5,"sessions":1}
{
  "ok": true,
  "service": "Team Sprint",
  "people": 5,
  "sessions": 1,
  "billing_units_per_session": 2,
  "total_usd": 120,
  "basis": "Published prices; no discounts assumed",
  "booking_confirmed": false
}
calculate_quote {"servi

## 7 Experiment design
The final matrix has 12 setups × 8 fixed scenarios × 2 repetitions = 192 live trials. A lead scenario contains two turns. All setups use the same base model and factual sources. Scenario order is shuffled with a recorded seed.

The study compares both persona bundles, private CoT-style prompting versus the direct condition, worked examples versus zero-shot, system versus in-message operational instructions, detailed versus reduced instructions, and changes to temperature, frequency penalty, presence penalty, token limit, and top-p. The top-p pair holds temperature at 1; other parameter ablations change one setting relative to Mira.

The protocol and safety constraints remain system-level in every condition. Top-k is unsupported by Chat Completions and is not sent. CoT-style prompting is a wording ablation; public logs do not prove internal reasoning. The recorded source snapshot and exact settings make the conditions inspectable.

In [7]:
from experiments import setups,CASES
display(pd.DataFrame([{'setup':s.name,'persona':s.persona,**s.generation.options(),
                      **asdict(s.prompt),'changed':s.changed} for s in setups()]))
print('Scenarios:',', '.join(c['id'] for c in CASES))

,setup,persona,model,temperature,top_p,frequency_penalty,presence_penalty,max_completion_tokens,cot,few_shot,instruction_role,detailed,changed
0,mira_default,mira,gpt-4.1-mini,0.6,1.0,0.0,0.2,650,False,True,system,True,Reference
1,sami_default,sami,gpt-4.1-mini,0.1,1.0,0.0,0.0,650,False,True,system,True,Persona bundle
2,mira_cot,mira,gpt-4.1-mini,0.6,1.0,0.0,0.2,650,True,True,system,True,Private step-by-step prompt only
3,mira_zero_shot,mira,gpt-4.1-mini,0.6,1.0,0.0,0.2,650,False,False,system,True,Remove worked examples only
4,mira_user_instructions,mira,gpt-4.1-mini,0.6,1.0,0.0,0.2,650,False,True,user,True,Move operational instructions to user message ...
5,mira_minimal,mira,gpt-4.1-mini,0.6,1.0,0.0,0.2,650,False,True,system,False,Replace operational rules with one generic ins...
6,mira_low_temp,mira,gpt-4.1-mini,0.1,1.0,0.0,0.2,650,False,True,system,True,Temperature only: 0.6 to 0.1
7,mira_frequency,mira,gpt-4.1-mini,0.6,1.0,0.8,0.2,650,False,True,system,True,Frequency penalty only: 0 to 0.8
8,mira_presence,mira,gpt-4.1-mini,0.6,1.0,0.0,0.8,650,False,True,system,True,Presence penalty only: 0.2 to 0.8
9,mira_short,mira,gpt-4.1-mini,0.6,1.0,0.0,0.2,180,False,True,system,True,Completion budget only: 650 to 180


Scenarios: service_price, team_quote, lead_continuity, feedback, unknown_policy, technical_advice, memory, integrity


## 8 Results and limitations
The analysis uses saved real executions. It separates functional checks from subjective naturalness. The original runs and scores remain unchanged; a separate scoring revision corrects a typographic-apostrophe false failure. The first development run is retained for comparison.

In [8]:
display(Markdown(Path('docs/experiment_results.md').read_text(encoding='utf-8')))
display(pd.read_csv('evidence/comparison.csv'))

# Experiment results

The final run contains **192 live trials**: 12 setups × 8 fixed scenarios × 2 repetitions. Lead collection uses two turns. Requests were shuffled with seed 503 and ran with four workers. All setups used `gpt-4.1-mini`. Both default personas passed 16/16 functional trials.

| Setup | Passed | Mean seconds per trial | Mean model calls | Mean final answer words |
|---|---:|---:|---:|---:|
| mira_cot | 16/16 | 3.60 | 2.25 | 42.4 |
| mira_default | 16/16 | 4.15 | 2.25 | 40.1 |
| mira_frequency | 16/16 | 3.98 | 2.25 | 39.5 |
| mira_low_p | 16/16 | 3.70 | 2.25 | 39.7 |
| mira_low_temp | 16/16 | 3.21 | 2.25 | 36.4 |
| mira_minimal | 14/16 | 3.50 | 2.25 | 42.0 |
| mira_p_control | 16/16 | 3.09 | 2.25 | 40.9 |
| mira_presence | 16/16 | 3.76 | 2.25 | 39.9 |
| mira_short | 15/16 | 3.60 | 2.19 | 33.1 |
| mira_user_instructions | 16/16 | 3.56 | 2.25 | 42.2 |
| mira_zero_shot | 15/16 | 4.18 | 2.31 | 50.7 |
| sami_default | 16/16 | 3.69 | 2.25 | 38.1 |

## What changed and what stayed fixed

- `mira_default` and `sami_default` compare complete persona bundles: role, style, and default generation settings. They do not isolate persona wording from temperature.
- The CoT-style condition adds a private step-by-step prompt. It returns the same brief public summary/answer format. No private chain-of-thought is collected, and the experiment cannot verify internal reasoning.
- Zero-shot removes the worked examples. Instruction placement moves only the operational instructions/examples between the system and current user message; the protocol, factual sources, and safety constraints stay system-level.
- The minimal condition replaces the detailed operational rules with one generic instruction, retaining the examples and common protocol.
- Temperature, frequency penalty, presence penalty, and token-budget conditions each change one setting relative to Mira. The top-p pair uses temperature 1 in both conditions and changes only top-p, from 1 to 0.5. Top-k is not accepted by this API and is never sent.

## Observations

The detailed instruction block mattered most on the multi-quote task. Both minimal-instruction repetitions substituted a four-person team for the requested five students in the two-session quote. The default instructions explicitly preserve the requested quantities, and both personas returned USD 240 with successful matching tool observations.

Zero-shot attempted to save a lead before an email was supplied in one trial. Validation rejected the attempt; after the next user message, the correct lead was saved. Worked examples avoided that premature action in the default trials. A 180-token completion limit truncated one comparison response after successful tools. The error was shown instead of presenting an incomplete answer as a pass.

The private step-by-step prompt, higher penalties, and top-p change did not improve functional success over the default on this case set. Several setups tied at 16/16, so there is no unique accuracy winner. The low-temperature Mira condition had shorter answers; observed timing differences are descriptive and may reflect provider/network scheduling under four-worker concurrency.

## Evidence and scoring limits

The checker requires completed turns, required successful tools, selected factual values, correct target quote arguments, exact lead contact fields, and no unrequested writes. Naturalness and overall helpfulness are assessed separately through examples, not assigned a fabricated numeric score. Passing these checks does not prove every sentence is correct.

The final run used 433 attempted API requests. Token totals in `comparison.csv` cover responses returned by the adapter; the truncated response is not included in those totals. Two repetitions and eight scenarios support an exploratory comparison, not statistical significance or universal reliability.

All trial inputs, actual replies, tool arguments, returned observations, model settings, usage, and latencies are under `evidence/experiments/trials/`. `plan.json` preserves the source snapshot used for the run. `manifest.json` hashes the original JSON artifacts. The original summary scored the minimal condition at 13/16 because a typographic apostrophe hid a correct refusal; `scoring_revision.json` documents the correction to 14/16 without changing responses.

The first full run is retained in `evidence/experiments_initial/`. It exposed the quote-quantity issue and a checker that incorrectly required the target quote to be last. `scoring_revision.json` there separates re-scoring from real model improvements. The second run followed the instruction fix; it is a fresh run, not a selection of only passing trials.

Quote totals are planning estimates using the published unit prices and an explicit grouping calculation. They do not establish availability or approval for multiple teams/sessions. Staff must confirm feasibility.


,setup,passed,trials,pass_rate,mean_seconds,mean_calls,mean_answer_words,prompt_tokens,completion_tokens
0,mira_cot,16,16,1.0000,3.599067,2.2500,42.4375,97635,1667
1,mira_default,16,16,1.0000,4.149544,2.2500,40.1250,97410,1614
2,mira_frequency,16,16,1.0000,3.984716,2.2500,39.5000,97294,1581
3,mira_low_p,16,16,1.0000,3.704744,2.2500,39.6875,97368,1593
4,mira_low_temp,16,16,1.0000,3.209989,2.2500,36.3750,97399,1569
5,mira_minimal,14,16,0.8750,3.498088,2.2500,42.0000,84947,1623
6,mira_p_control,16,16,1.0000,3.086429,2.2500,40.9375,97312,1607
7,mira_presence,16,16,1.0000,3.763215,2.2500,39.9375,97403,1651
8,mira_short,15,16,0.9375,3.599870,2.1875,33.1250,94335,1409
9,mira_user_instructions,16,16,1.0000,3.561452,2.2500,42.2500,97772,1683


## 9 Automated verification
These tests exercise actual LangGraph transitions and local functions with scripted model outputs, including tool errors, memory, exact observations, deduplication, limits, and final-answer parsing. They establish orchestration behavior, not live model quality.

In [9]:
result=subprocess.run([sys.executable,'-m','pytest','tests','-q'],capture_output=True,text=True)
print(result.stdout)
if result.returncode: print(result.stderr)
assert result.returncode==0

.................................                                        [100%]
33 passed in 5.69s



## 10 Optional live execution
Configure a key locally in `.env`. Set `RUN_LIVE=true` to run this one example with real API calls. Repeating the full matrix is a separate CLI command so Run All does not silently repeat paid experiments. The fresh-kernel verification disables these flags.

In [10]:
if os.getenv('RUN_LIVE','false').lower()=='true':
    live=ReActAgent(OpenAIModel(),persona='mira')
    reply,trace=live.respond('What does Prototype Clinic cost and how long is it?',return_trace=True)
    print(reply)
    display(trace['events'])
else:
    print('Live calls disabled. Saved real API trials are shown above.')

Live calls disabled. Saved real API trials are shown above.


## 11 Gradio interface
The interface lets the user choose a persona, adjust generation/prompt settings, chat, and inspect actual actions and observations. Switching personas clears the conversation and restores that persona's defaults. Each callback receives only its session's history.

In [11]:
from app import build_demo,THEME,CSS
if os.getenv('RUN_UI','false').lower()=='true':
    demo=build_demo()
    demo.queue().launch(server_name='127.0.0.1',server_port=7861,share=False,
                        prevent_thread_lock=True,theme=THEME,css=CSS)
else:
    print('Notebook server launch disabled. Run python app.py for the local interface.')

C:\Users\jijos\Desktop\EECE 503P\C4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook server launch disabled. Run python app.py for the local interface.


## 12 Reflection

## Which persona gave the most helpful or natural results

Mira is the better entry point for students who need help choosing a service. Her advice is conversational and ends with a practical next step. Sami is more useful when a student already has a technical blocker. In the saved intermittent-test example, Sami asks for a reproducible test case and relevant environment constraints; Mira asks for the current test code and a description of the problem. Both stay within coaching rather than doing the student's work.

Both defaults passed 16/16 functional trials, so this preference comes from the saved answer styles and their intended roles, not an accuracy difference. The difference is clearest in technical preparation advice and less obvious in routine price or contact collection replies. There was no independent human preference study.

## Which prompt and configuration combination performed best

There was no unique functional winner: both default personas and several ablations passed 16/16. For normal service intake, the practical choice is Mira with the detailed instructions and worked examples in the system message, temperature 0.6, top-p 1, frequency penalty 0, presence penalty 0.2, and a 650-token completion limit. For technical scoping, Sami uses the same protocol and instructions with temperature 0.1 and both penalties 0.

The defaults were retained because they serve different roles and handle the recorded tasks without unnecessary tuning. Low-temperature Mira also passed and produced shorter answers on average. Timing differences are too small and noisy to establish a best configuration. Raising penalties and changing top-p added no functional benefit on these cases. A 180-token limit failed once on a long comparison, so it is too restrictive for the full task set.

The CoT-style condition also passed 16/16, but did not improve success over the default. It asks for private step-by-step thinking while retaining the same short public output format. This tests the effect of that prompt wording; it cannot establish what internal reasoning occurred.

## How well did the agent reason and use tools

The graph explicitly alternates model decisions, tool execution, and returned observations. For the comparison of two Team Sprint sessions for five students, the saved default trace reads the service catalog, calculates comparison quotes, requests the exact five-person two-session quote, and then returns USD 240. The tool observation includes the people, sessions, billing units, and total. The final answer is parsed only after the observations are returned with matching call IDs.

Lead collection demonstrates multi-turn context: after a name and service are supplied, the assistant asks for the missing email and reuses the earlier details. Feedback and unknown-policy questions produce actual local records. A tool call is not counted as success unless its observation has ok=true. Duplicate calls within a turn reuse the original observation.

The first experiment run exposed a genuine mistake: some answers used four people for the requested five-person total. The grouping capacity was being confused with the user's quantity. The revised instructions preserve the exact quantities, and both default personas passed the repeated quote case in the final run. This is improvement on a fixed development benchmark, not evidence of performance on unseen tasks.

## Did the instructions block change behavior noticeably

Yes, particularly on quantity preservation and premature actions. With the reduced operational instructions, both final quote repetitions substituted four people for the requested five-person two-session estimate. The detailed-instruction defaults used the correct quantities and returned USD 240. The minimal condition scored 14/16 after a documented scoring correction.

Removing the worked examples caused one premature lead-save attempt before an email was supplied. The validator rejected it, so no invented contact was saved, and the later email message completed the lead correctly. The default examples explicitly show asking for the missing email instead of calling the writing tool. Zero-shot scored 15/16; the default scored 16/16.

Moving operational instructions into the current user message did not reduce success in this small run. The protocol and safety constraints stayed at system level, so this does not show that arbitrary user-level instructions are equally robust. The test set contains only eight scenarios with two repetitions.

## What were the biggest implementation challenges

The main challenge was separating a model's requested action from a completed action. The graph must validate arguments, execute the function, return the exact observation, and let the model continue before accepting a final answer. Error handling also has to retain record IDs when a later model call fails after a successful write.

The second challenge was keeping the comparisons interpretable. Persona bundles intentionally change both wording and default settings, while the within-Mira ablations change one factor at a time. A separate top-p control keeps temperature fixed. The first scorer mistakenly checked only the last quote in a comparison; another check missed a refusal containing a typographic apostrophe. Both corrections are documented separately, and the original scores and responses remain available.

The final limitation is evaluation scope. Functional checks can verify selected facts, arguments, and saved records, but cannot prove naturalness, complete factual accuracy, or private reasoning quality. Two repetitions provide an exploratory comparison. Quotes are planning estimates using unit prices and grouping assumptions; they do not confirm availability. Local JSONL logging, finite history, and model-dependent interpretation remain classroom-prototype limits.


## 13 References
- C4 assignment, chapter notes, and lecture deck supplied with the course.
- [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api).
- [OpenAI Chat Completions parameters](https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create).
- [GPT 4.1 mini model documentation](https://developers.openai.com/api/docs/models/gpt-4.1-mini).

The complete requirement mapping, strict self-assessment, and remaining limits are in `docs/self_assessment.md`.